In [ ]:
import logging
from scrapling.fetchers import AsyncDynamicSession, AsyncStealthySession
from playwright.async_api import Page

logging.getLogger("scrapling").setLevel(logging.WARNING)

In [ ]:
EMAIL = "admin@example.com"
PASSWORD = "password"

# 1. Normal login
print("=== Normal Login ===")
async def do_login(page: Page):
    await page.fill('input[type="email"]', EMAIL)
    await page.fill('input[type="password"]', PASSWORD)
    await page.click('button[type="submit"]')
    await page.wait_for_load_state("networkidle")

async with AsyncDynamicSession(headless = False) as session:
    login_page = await session.fetch(
        "https://www.scrapingcourse.com/login",
        page_action = do_login,
    )
    print("Current URL:", login_page.url)  # should no longer be /login if successful

In [ ]:
# 2. Login with CSRF token
# The only case where we'd need to manually handle a CSRF token is if we were making raw HTTP
# requests (e.g., with requests or httpx) without a real browser. With Playwright, the browser
# handles it for you.
print("\n=== Login with CSRF Token (handled by Playwright) ===")
async with AsyncDynamicSession(headless = True) as session:
    login_url = "https://www.scrapingcourse.com/login/csrf"
    login_page = await session.fetch(
        login_url,
        page_action = do_login,
    )
    print("Current URL:", login_page.url)

In [ ]:
# 3. Login with anti-bot measures
# Just switching to a stealthy session should bypass most anti-bot measures, as it mimics a real
# browser more closely.
print("\n=== Login with Anti-Bot Measures ===")
async with AsyncStealthySession(headless = True) as session:
    login_url = "https://www.scrapingcourse.com/login/cf-antibot"
    login_page = await session.fetch(
        login_url,
        page_action = do_login,
    )
    print("Current URL:", login_page.url)

In [ ]:
# 4. Login with both Cloudflare Turnstile
print("\n=== Login with Cloudflare Turnstile ===")
async def do_login_with_turnstile(page: Page):
    await page.wait_for_function(
        'document.querySelector(\'[name="cf-turnstile-response"]\')?.value?.length > 0'
    )
    await page.fill('input[name="email"]', EMAIL)
    await page.fill('input[name="password"]', PASSWORD)
    await page.click('button[type="submit"]')
    await page.wait_for_load_state("networkidle")

async with AsyncStealthySession(headless=True) as session:
    login_url = "https://www.scrapingcourse.com/login/cf-turnstile"
    login_page = await session.fetch(
        login_url,
        page_action = do_login_with_turnstile,
    )
    print("Current URL:", login_page.url)

In [ ]:
# 5. Do both the Cloudflare and the anti-bot challenges
# This is the most realistic scenario, as many sites will have multiple layers of protection. Again,
# using a stealthy session should allow us to bypass both.
# Note that the advanced stealth features are given by real_chrome=True and solve_cloudflare=True, 
# so there was probably no need to implement before the do_login_with_turnstile function, but I
# wanted to show how you could handle a custom page action if needed.
print("\n=== Login with Both Cloudflare and Anti-Bot Measures ===")
async with AsyncStealthySession(headless = False, real_chrome = True,
                                solve_cloudflare = True) as session:
    url = "https://www.scrapingcourse.com/cloudflare-challenge"
    url_page = await session.fetch(url,)
    print("Current URL:", url_page.url)
    page_text = url_page.css("h2.challenge-title")[0].text.strip()
    print("Page Text:", page_text)

    url = "https://www.scrapingcourse.com/antibot-challenge"
    url_page = await session.fetch(url)
    print("Current URL:", url_page.url)
    page_text = url_page.css("h2.challenge-title")[0].text.strip()
    print("Page Text:", page_text)